<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/3_Analisis_y_visualizacion_datos/3_Casos_Especiales_Importacion_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# 🛢️**Casos especiales importación de datos**

**Objetivos de Aprendizaje:**

Al finalizar este notebook, serás capaz de:
1. Conectar y extraer datos desde bases de datos SQL (SQLite) y archivos semi-estructurados como JSON.
2. Llamar todos los archivos dentro de una carpeta y concatenarlos.
3. Cargar una base muy pesada por chunks.

---

## Importar librerías

In [1]:
import os
import numpy as np
import pandas as pd

## Mejorar visualización de los dataframes

In [2]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

## Establecer la ruta de los datasets

In [3]:
from google.colab import drive, files
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
path = '/content/drive/MyDrive/2026-i-curso-ia-para-economia/datasets'

In [5]:
# Para establecer el directorio de los archivos
os.chdir(path)

## Importando Datos de Diversas Fuentes

Los datos económicos no siempre vienen en un `csv` o `Excel`. A menudo, necesitamos interactuar con sistemas más complejos como bases de datos o formatos web como JSON.

### Importación desde una Base de Datos SQL

Las bases de datos son la columna vertebral de cualquier sistema de información. Bancos, entidades gubernamentales y empresas almacenan sus registros en bases de datos relacionales. Aprender a extraer datos de ellas es una habilidad crucial.

**¿Qué es una Base de Datos Relacional?**🗄️

Imagina que administras una tienda. Podrías registrar todas tus ventas en una **gigantesca hoja de Excel**. Por cada venta, anotarías el nombre del cliente, su email, el nombre del producto, su precio, la fecha, etc.

Pronto notarías los **problemas**:

- **Redundancia:** Si Ana Pérez compra 10 veces, anotas su nombre y email 10 veces.

- **Riesgo de Errores**: En una de esas 10 veces, podrías escribir "Ana Peres". Ahora, para el sistema, son dos clientas diferentes, y tus análisis serán incorrectos.

- **Ineficiencia:** El archivo se vuelve enorme y lento.

**Una base de datos relacional resuelve esto de forma elegante.** En lugar de una tabla gigante, creas tablas que se relacionan entre sí:

- **Tabla Clientes:** Solo guarda información de clientes. Cada cliente tiene un "id" único: cliente_id, el cuál podría ser su número de cédula.

- **Tabla Productos:** Solo guarda información de productos. Cada producto tiene un id único: producto_id.

- **Tabla Ventas:** Registra un único id para la venta (venta_id), el cual podría ser el número de factura, la fecha de venta, la cantidad de productos y lo más importante: cliente_id y producto_id. En lugar de repetir todos los datos de clientes y productos vendidos, esta tabla solo registra los ids de estos.

**La "relación" es el vínculo que se crea mediante estos IDs.** La tabla Ventas no sabe el nombre del cliente, pero sabe su cliente_id, y con este puede ir a buscar los detalles en la tabla Clientes.

**En resumen, una base de datos relacional es un sistema para almacenar datos en tablas separadas y conectadas por llaves, lo que elimina la redundancia, garantiza la consistencia y permite cruzar información de forma muy poderosa.**

![Ejemplo Base de Datos Realcional](https://drive.google.com/uc?id=1o0XV423UuWkbFzS6TUS_BfPrc12Bs2ln)

**¿Qué es SQL?** 🗣️

**SQL (Structured Query Language) es el idioma que se usa para pedirle al sistema que busque, filtre y combine información para generar reportes.**

No es un lenguaje de programación como Python, sino un lenguaje de consulta. **Su objetivo es solicitar, manipular y gestionar datos dentro de una base de datos relacional.**

La estructura de una petición básica en SQL es muy intuitiva y se parece a cómo pensarías la pregunta en español. Las "palabras clave" o **comandos más básicos** son:

- **SELECT:** Elige las columnas que quieres ver.

- **FROM:** Indica de qué tabla quieres sacar los datos.

- **WHERE:** Aplica un filtro a las filas (observaciones) para obtener solo las que cumplen una condición.

Pregunta de negocio simple: **"Quiero ver las ventas donde se ha vendido más de 5 unidades."**

In [ ]:
SELECT * # Tráeme todas las columnas
FROM Ventas # de la tabla Ventas
WHERE cantidad > 5; # pero solo las filas (registros) donde se vendieron más de 5 unidades

Aprender SQL es fundamental para un economista porque te da acceso directo a las fuentes primarias de datos en empresas y gobiernos, sin depender de que alguien más te exporte un archivo .csv.

Para el ejemplo, usaremos **SQLite**, un Sistema Gestor de Bases de Datos Relacionales sencillo que viene incluido con Python.

**Nota:** Un **Sistema Gestor de Bases de Datos Relacionales** (SGBDR), o RDBMS por sus siglas en inglés, es el **software especializado que te permite crear, consultar, actualizar y administrar una base de datos relacional.** Se encarga de la seguridad, la integridad, la eficiencia de las consultas y el acceso concurrente, permitiendo que los datos sean un activo fiable y utilizable para una organización.

Otros SGBDR son PostgreSQL y MySQL.

In [6]:
import sqlite3

In [7]:
# Paso 1: Establecer una conexión con la base de datos SQLite
conn = sqlite3.connect('economia_colombia.db')

In [8]:
# Paso 2: Extraer los datos usando una consulta SQL
# Consulta: Selecciona todas las columnas de la tabla 'indicadores_macro' (trae todos los datos)

query = "SELECT * FROM indicadores_macro"
df_from_sql = pd.read_sql_query(query, conn)

print("Datos extraídos de la base de datos:")
display(df_from_sql)

Datos extraídos de la base de datos:


,año,tasa_inflacion,tasa_desempleo
0,2020,1.61,15.90
1,2021,5.62,13.70
2,2022,13.12,11.20
3,2023,9.28,10.20


In [9]:
# Otra consulta: "Selecciona todas las columnas de la tabla 'indicadores_macro' donde el año sea mayor a 2021"

query2 = "SELECT * FROM indicadores_macro WHERE año > 2021"
df_from_sql2 = pd.read_sql_query(query2, conn)

print("Datos extraídos de la base de datos:")
display(df_from_sql2)

Datos extraídos de la base de datos:


,año,tasa_inflacion,tasa_desempleo
0,2022,13.12,11.20
1,2023,9.28,10.20


In [10]:
# Paso 3: cerrar la conexión
conn.close()

---

### Importación de Archivos JSON

**JSON (JavaScript Object Notation)** es el lenguaje de la web. Muchas APIs y fuentes de datos abiertos (como las del gobierno) entregan la información en este formato, que es flexible pero puede ser anidado.

Vamos a utilizar un ejemplo que simula datos de población y PIB per cápita para Colombia.

In [11]:
import json

Cargamos el archivo .json

In [12]:
# Esta línea lee el archivo JSON y lo convierte en un objeto de Python
json_data=json.load(open('ejemplo_poblacion_pib_pc.json','r'))
json_data

{'pais': 'Colombia',
 'datos_poblacionales': [{'año': 2014,
   'poblacion_total': 49296000,
   'pib_per_capita': 5856},
  {'año': 2015, 'poblacion_total': 49598000, 'pib_per_capita': 5995},
  {'año': 2016, 'poblacion_total': 49738700, 'pib_per_capita': 6058},
  {'año': 2017, 'poblacion_total': 50157000, 'pib_per_capita': 6126},
  {'año': 2018, 'poblacion_total': 50482900, 'pib_per_capita': 6293},
  {'año': 2019, 'poblacion_total': 50921000, 'pib_per_capita': 6335},
  {'año': 2020, 'poblacion_total': 51034000, 'pib_per_capita': 6498},
  {'año': 2021, 'poblacion_total': 51230000, 'pib_per_capita': 6550},
  {'año': 2022, 'poblacion_total': 51520000, 'pib_per_capita': 6630},
  {'año': 2023, 'poblacion_total': 52085000, 'pib_per_capita': 6710}]}

En pandas, podemos crear un dataframe directamente a partir de un objeto tipo JSON, ya que este archivo es equivalente a un diccionario de Python. Sin embargo, puede ocurrir que no obtengamos la tabla que queremos.

In [13]:
data = pd.DataFrame(json_data)
data

,pais,datos_poblacionales
0,Colombia,"{'año': 2014, 'poblacion_total': 49296000, 'pi..."
1,Colombia,"{'año': 2015, 'poblacion_total': 49598000, 'pi..."
2,Colombia,"{'año': 2016, 'poblacion_total': 49738700, 'pi..."
3,Colombia,"{'año': 2017, 'poblacion_total': 50157000, 'pi..."
4,Colombia,"{'año': 2018, 'poblacion_total': 50482900, 'pi..."
5,Colombia,"{'año': 2019, 'poblacion_total': 50921000, 'pi..."
6,Colombia,"{'año': 2020, 'poblacion_total': 51034000, 'pi..."
7,Colombia,"{'año': 2021, 'poblacion_total': 51230000, 'pi..."
8,Colombia,"{'año': 2022, 'poblacion_total': 51520000, 'pi..."
9,Colombia,"{'año': 2023, 'poblacion_total': 52085000, 'pi..."


Para convertirlo a DataFrame, a menudo es necesario "normalizar" los datos anidados.

En este ejemplo, nos interesa la lista que está bajo la llave "datos_poblacionales", por lo que en lugar de convertir directamente a un dataframe, debemos utilizar json_normalize e indicar el record_path que queremos.

In [14]:
df_from_json = pd.json_normalize(json_data, record_path='datos_poblacionales')
df_from_json

,año,poblacion_total,pib_per_capita
0,2014,49296000,5856
1,2015,49598000,5995
2,2016,49738700,6058
3,2017,50157000,6126
4,2018,50482900,6293
5,2019,50921000,6335
6,2020,51034000,6498
7,2021,51230000,6550
8,2022,51520000,6630
9,2023,52085000,6710


El código anterior solo extrae la tabla anidada de datos_poblacionales. ¿Pero qué pasa con la información del nivel superior, como "pais": "Colombia"?

Para eso, json_normalize tiene el parámetro meta, en el cual podemos indicar qué otras llaves del nivel superior queremos añadir como columnas en nuestro dataFrame final.

In [15]:
df_completo = pd.json_normalize(json_data, record_path='datos_poblacionales', meta=['pais'])
df_completo

,año,poblacion_total,pib_per_capita,pais
0,2014,49296000,5856,Colombia
1,2015,49598000,5995,Colombia
2,2016,49738700,6058,Colombia
3,2017,50157000,6126,Colombia
4,2018,50482900,6293,Colombia
5,2019,50921000,6335,Colombia
6,2020,51034000,6498,Colombia
7,2021,51230000,6550,Colombia
8,2022,51520000,6630,Colombia
9,2023,52085000,6710,Colombia


## Llamar todos los archivos dentro de una carpeta y concatenarlos

**Ejemplo:** Eres analista del DANE en la Coordinación de Comercio Internacional y cuentas con una carpeta que contiene los archivos de exportaciones de bienes correspondientes a julio, agosto y septiembre de 2024. Necesitas consolidarlos en un único archivo del tercer trimestre de 2024, pero no deseas cargar cada archivo de forma manual, sino que el programa procese automáticamente todos los archivos Excel ubicados en la carpeta que especifiques.

In [16]:
import glob

**Nota:** La librería glob se usa para buscar y listar archivos en carpetas que cumplen un patrón en su nombre (comodines como *, ?). Es muy útil cuando tienes muchos archivos CSV, Excel, TXT, etc. en una carpeta y quieres procesarlos en un solo paso.

In [17]:
# Buscar todos los archivos Excel en la carpeta
archivos = glob.glob('expo_bienes_2024/*.xlsx')

# Lista para guardar DataFrames temporales
lista_dfs = []

for archivo in archivos:
    try:
        temp = pd.read_excel(archivo)
        lista_dfs.append(temp)

        # Verificar columnas para identificar si algún archivo tiene más de las que debería
        if len(temp.columns) > 72:
            print(archivo, len(temp.columns))

    except Exception as e:
        print(f"Error leyendo {archivo}: {e}")

# Concatenar todos los DataFrames en uno solo
df_expo = pd.concat(lista_dfs, ignore_index=True)
df_expo.head(3)

,FECHA_PROCESO,NUMERO_SERIE,OFICINA,COD_ADUANA_DESPACHO,ADUANA_DESPACHO,TIPO_IDENT,NIT_EXPORTADOR,TIPO_USUARIO,COD_USUARIO,CLASE_EXPORTADOR,COD_DPTO_EXPORTADOR,COD_PAIS_DESTINO_NUM,COD_PAIS_DESTINO_ALF,COD_PAIS_DESTINO,PAIS_DESTINO_FINAL,CIUDAD_DESTINATARIO,NUM_SOLICITUD_AUTO_EMBARQUE,TIPO_DECLARACION,TIPO_DESPACHO,COD_LUGAR_SALIDA_NUM,COD_LUG_SALIDA_ALF,COD_REGION_PROCEDENCIA,REGION_PROCEDENCIA,NUM__DECLA_EXPORTACION_ANT,FECH_DECLA_EXPORTACION_ANT,NUM_DECLARACION_PRECEDENTE,FECH_DECLA_PRECEDENTE,COD_MODALIDAD_PRECEDENTE,COD_MONEDA_TRANSACCION,COD_MODO_TRANSPORTE,MODO_TRANSPORTE,BANDERA,COD_NACIONALIDAD_BANDERA,NACIONALIDAD_BANDERA,COD_REGIMEN_CAN,COD_MODALIDAD_EXPORTACION,MODALIDAD_EXPORTACION,FORMA_PAGO,COD_TIPO_EMBARQUE,TIPO_DE_EMBARQUE,COD_TIPO_DATOS,TIPO_DE_DATOS,TIPO_CERTIFICADO_ORIGEN,SISTEMAS_ESPECIALES,COD_EXPORTACION_TRANSITO,EXPORTACION_EN_TRANSITO,SUBPARTIDA,COD_REGION_ORIGEN,REGION_DE_ORIGEN,COD_UNIDAD_FISICA_NUM,COD_UNIDAD_FISICA_ALF,UNIDAD_FISICA,CANTIDAD_UNIDADES_FISICAS,PESO_BRUTO_KGS,PESO_NETO_KGS,VALOR_FOB_USD,VALOR_FOB_PESOS,VLR_SERIE_AGREGADO_NAL_USD,VALOR_SERIE_FLETES_USD,VALOR_SERIE_SEGUROS_USD,VLR_SERIE_OTROS_GASTOS_USD,COD_ADUANA_SALIDA,ADUANA_SALIDA,FECHA_SOLICITUD_AUTO_EMBARQUE,NUMERO_FORMULARIO,FECHA_DECLARACION_EXPORTACION,RAZON_SOCIAL_EXPORTADOR,DIREC_EXPORTADOR,NIT_DECLARANTE,RAZON_SOCIAL_DECLARANTE,RAZON_SOCIAL_DESTINATARIO,DOMICILIO_DESTINATARIO
0,202407,7,99,89,Aduanas de Cúcuta,2,"901,333,018.00",36,0,2,54001,850,VEN,VE,VENEZUELA (REPÚBLICA BOLIVARIANA DE),MERIDA,"6,027,733,007,951.00",1,Inicial,89,CUC,54,NORTE DE SANTANDER,0,0,0,0,NaN,USD,3,Terrestre (carretero),13,AF,AFGANISTÁN,1,198,Exportación definitiva de mercancías de fabric...,1,1,Unico,1,Definitivos al embarque,8,NO,N,NO,8212102000,54,NORTE DE SANTANDER,11,U,Unidades o artículos,74.00,"3,509.35","3,498.00","5,476.00","21,983,730.56",0.00,0.00,0.00,0.00,89,Aduanas de Cúcuta,202407,6007739047723,20240729,C.I. FERGUTECH SAS,BG E 2 3 ZN FRANCA,"901,557,278.00",BERZATEX SAS.,COMERCIALIZADORA Y DISTRIBUIDORA LA CHINA C.A,AV BOLIVAR LC NO S/N ZONA ENTRE AV 16 Y 18
1,202407,6,99,89,Aduanas de Cúcuta,2,"901,333,018.00",36,0,2,54001,850,VEN,VE,VENEZUELA (REPÚBLICA BOLIVARIANA DE),MERIDA,"6,027,733,007,951.00",1,Inicial,89,CUC,54,NORTE DE SANTANDER,0,0,0,0,NaN,USD,3,Terrestre (carretero),13,AF,AFGANISTÁN,1,198,Exportación definitiva de mercancías de fabric...,1,1,Unico,1,Definitivos al embarque,8,NO,N,NO,3401300000,54,NORTE DE SANTANDER,33,KG,Kilogramo,515.00,516.67,515.00,780.00,"3,131,356.80",0.00,0.00,0.00,0.00,89,Aduanas de Cúcuta,202407,6007739047723,20240729,C.I. FERGUTECH SAS,BG E 2 3 ZN FRANCA,"901,557,278.00",BERZATEX SAS.,COMERCIALIZADORA Y DISTRIBUIDORA LA CHINA C.A,AV BOLIVAR LC NO S/N ZONA ENTRE AV 16 Y 18
2,202407,5,99,89,Aduanas de Cúcuta,2,"901,333,018.00",36,0,2,54001,850,VEN,VE,VENEZUELA (REPÚBLICA BOLIVARIANA DE),MERIDA,"6,027,733,007,951.00",1,Inicial,89,CUC,54,NORTE DE SANTANDER,0,0,0,0,NaN,USD,3,Terrestre (carretero),13,AF,AFGANISTÁN,1,198,Exportación definitiva de mercancías de fabric...,1,1,Unico,1,Definitivos al embarque,8,NO,N,NO,3809910000,54,NORTE DE SANTANDER,33,KG,Kilogramo,"1,056.00","1,059.42","1,056.00","1,600.00","6,423,296.00",0.00,0.00,0.00,0.00,89,Aduanas de Cúcuta,202407,6007739047723,20240729,C.I. FERGUTECH SAS,BG E 2 3 ZN FRANCA,"901,557,278.00",BERZATEX SAS.,COMERCIALIZADORA Y DISTRIBUIDORA LA CHINA C.A,AV BOLIVAR LC NO S/N ZONA ENTRE AV 16 Y 18


In [18]:
df_expo.shape

(20532, 72)

In [19]:
# Para comprobar los meses que quedaron en el dataframe
df_expo['FECHA_PROCESO'].value_counts(dropna=False)

,count
FECHA_PROCESO,
202408,7557
202409,7548
202407,5427


## Cargar una base muy pesada (varias gigas) por chunks

**Ejemplo**: Trabajas en la Dirección de Estudios Económicos de la Superintendencia de Sociedades y necesitas cargar y analizar la base de datos de las 10.000 empresas más grandes del país. Sin embargo, la carga completa está tardando varias horas y tu jefe te ha solicitado el análisis para las 4 de la tarde. Para optimizar el proceso y reducir significativamente el tiempo, decides realizar la carga por chunks.

In [20]:
# Importamos la librería time que nos ayudará a medir el tiempo de ejecución
from time import time

In [21]:
tiempo_inicio = time()
df_ssc = pd.read_csv('Supersociedades_2023.csv', encoding ='utf-8', dtype=str)
df_ssc_reducido = df_ssc[['NIT', 'RAZON_SOCIAL', 'DEPARTAMENTO', 'MUNICIPIO',
                            'CIIU', 'DESCRIPCION_CIIU']]
tiempo_termino = time()
tiempo_carga_completa= tiempo_termino - tiempo_inicio
print(f"Tiempo carga completa: {tiempo_carga_completa:.6f} segs")

Tiempo carga completa: 0.469907 segs


In [22]:
df_ssc_reducido.head(3)

,NIT,RAZON_SOCIAL,DEPARTAMENTO,MUNICIPIO,CIIU,DESCRIPCION_CIIU
0,899999068,ECOPETROL S.A,"BOGOTÁ, D.C.","BOGOTÁ, D.C.",0610,Extracción de petróleo crudo
1,900112515,REFINERIA DE CARTAGENA S.A.,BOLÍVAR,CARTAGENA DE INDIAS,1921,Fabricación de productos de la refinación del ...
2,890100577,AEROVIAS DEL CONTINENTE AMERICANO S.A. AVIANCA,"BOGOTÁ, D.C.","BOGOTÁ, D.C.",5111,Transporte aéreo nacional de pasajeros


In [23]:
df_ssc_reducido.shape

(10000, 6)

In [24]:
tiempo_inicio = time()
df_chunk = pd.read_csv('Supersociedades_2023.csv', encoding ='utf-8', chunksize=1000)
chunk_list = []
for chunk in df_chunk:
    chunk_reducido = chunk[['NIT', 'RAZON_SOCIAL', 'DEPARTAMENTO', 'MUNICIPIO',
                            'CIIU', 'DESCRIPCION_CIIU']]

    # Añadir los datos a la lista de objetos
    chunk_list.append(chunk_reducido)

# Concatenar los datos filtrados en un DataFrame
df_ssc_reducido2 = pd.concat(chunk_list)
tiempo_termino = time()
tiempo_chunks = tiempo_termino - tiempo_inicio
print(f"Tiempo carga por chunks: {tiempo_chunks:.6f} segs")

Tiempo carga por chunks: 0.106753 segs


In [25]:
df_ssc_reducido2.head()

,NIT,RAZON_SOCIAL,DEPARTAMENTO,MUNICIPIO,CIIU,DESCRIPCION_CIIU
0,899999068,ECOPETROL S.A,"BOGOTÁ, D.C.","BOGOTÁ, D.C.",610,Extracción de petróleo crudo
1,900112515,REFINERIA DE CARTAGENA S.A.,BOLÍVAR,CARTAGENA DE INDIAS,1921,Fabricación de productos de la refinación del ...
2,890100577,AEROVIAS DEL CONTINENTE AMERICANO S.A. AVIANCA,"BOGOTÁ, D.C.","BOGOTÁ, D.C.",5111,Transporte aéreo nacional de pasajeros
3,830095213,ORGANIZACIÓN TERPEL S.A.,"BOGOTÁ, D.C.","BOGOTÁ, D.C.",4661,"Comercio al por mayor de combustibles sólidos,..."
4,900156264,NUEVA EMPRESA PROMOTORA DE SALUD S.A.,"BOGOTÁ, D.C.","BOGOTÁ, D.C.",8430,Actividades de planes de seguridad social de a...


In [26]:
df_ssc_reducido2.shape

(10000, 6)